In [0]:
# Databricks notebook source

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_table = "fintech_lakehouse.bronze.orders_raw"
silver_events_table = "fintech_lakehouse.silver.order_events_clean"
silver_current_table = "fintech_lakehouse.silver.orders_current"


# To Read Bronze events

bronze_df = spark.table(bronze_table)

# Clean and validate records

clean_events = (
    bronze_df
    .withColumn("order_id", F.col("order_id").cast("long"))
    .withColumn("customer_id", F.col("customer_id").cast("long"))
    .withColumn("amount", F.col("amount").cast("double"))
    .withColumn(
        "sequence_number",
        F.col("sequence_number").cast("long")
    )
    .withColumn(
        "event_timestamp",
        F.to_timestamp("event_time")
    )
    .withColumn(
        "event_type",
        F.upper(F.trim(F.col("event_type")))
    )
    .withColumn(
        "status",
        F.upper(F.trim(F.col("status")))
    )
    .filter(F.col("event_id").isNotNull())
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("event_timestamp").isNotNull())
    .filter(F.col("amount") >= 0)
    .filter(
        F.col("event_type").isin(
            "INSERT",
            "UPDATE",
            "CANCEL"
        )
    )
)

# Remove duplicate event IDs

event_window = (
    Window
    .partitionBy("event_id")
    .orderBy(
        F.col("sequence_number").desc(),
        F.col("ingested_at").desc()
    )
)

clean_events = (
    clean_events
    .withColumn(
        "event_rank",
        F.row_number().over(event_window)
    )
    .filter(F.col("event_rank") == 1)
    .drop("event_rank")
)


# Save the complete cleaned event history

(
    clean_events.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_events_table)
)

print(f"Clean event history created: {silver_events_table}")



# Find the latest event for every order

order_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("sequence_number").desc(),
        F.col("event_timestamp").desc(),
        F.col("ingested_at").desc()
    )
)

latest_orders = (
    clean_events
    .withColumn(
        "order_rank",
        F.row_number().over(order_window)
    )
    .filter(F.col("order_rank") == 1)
    .drop("order_rank")
    .withColumn(
        "is_cancelled",
        F.col("event_type") == "CANCEL"
    )
    .withColumn(
        "silver_updated_at",
        F.current_timestamp()
    )
)



# Create the current-state table on the first run

if not spark.catalog.tableExists(silver_current_table):

    (
        latest_orders.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_current_table)
    )

    print(f"Created Silver table: {silver_current_table}")

else:
    # Apply CDC updates on subsequent runs

    target = DeltaTable.forName(
        spark,
        silver_current_table
    )

    (
        target.alias("target")
        .merge(
            latest_orders.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenMatchedUpdateAll(
            condition="""
                source.sequence_number > target.sequence_number
                OR (
                    source.sequence_number = target.sequence_number
                    AND source.event_timestamp > target.event_timestamp
                )
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Updated Silver table: {silver_current_table}")



# Display current order state

display(
    spark.table(silver_current_table)
    .orderBy(F.col("silver_updated_at").desc())
)



# Silver data-quality summary

display(
    spark.sql(f"""
        SELECT
            COUNT(*) AS current_orders,
            COUNT(DISTINCT order_id) AS unique_orders,
            SUM(
                CASE WHEN is_cancelled THEN 1 ELSE 0 END
            ) AS cancelled_orders,
            ROUND(AVG(amount), 2) AS average_order_value
        FROM {silver_current_table}
    """)
)